In [ ]:
import pandas as pd
import json
import mne
import numpy as np
import mne
import os
import keras
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [ ]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [ ]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [ ]:
nan_counts = df.isna().sum()
print(nan_counts)

participant_id       0
subject_id           0
group                0
updrs_part_iii       5
updrs_total          5
moca                 4
age                  0
sex                  0
disease_duration    28
ledd                29
pigd_score          31
td_score            31
ctt                  9
dtype: int64


In [ ]:
print(df.shape)

(144, 13)


In [ ]:
missing_ids = []

for i in range(1, 145):
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

    # Check if file exists; if not, store or print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

In [ ]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_58/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [ ]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [ ]:

def get_eeg_signal(sub_id, task="walk", band="full",target_sfreq=256,duration=2.0, notch_freq=50.0,
    reject_threshold=0.00028,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, and segments EEG data.
    Returns float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }

    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")

    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & isolate EEG
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter (only if applicable to selected band range)
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency (e.g., 128 Hz)
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. Epoching with Artifact Rejection
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw,
        events=events,
        tmin=0,
        tmax=duration - (1 / target_sfreq),  # Exactly 256 samples at 128 Hz
        baseline=None,
        reject=reject_criteria,
        preload=True,
        verbose=False
    )

    # Extract array and cast to float32 to save RAM
    signal = epochs.get_data().astype(np.float32)

    return signal


# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


Beta Walk Signal shape (N, C, T): (121, 60, 512)
Full Rest Signal shape (N, C, T): (121, 60, 512)


In [ ]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [ ]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [ ]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []

    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids

    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)

            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)

        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")

    return X_hc, X_pd


In [ ]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]

    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)

    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active

        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break

        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0

                available = maj_counts[i] - allocations[i]
                take = min(share, available)

                allocations[i] += take
                remaining_target -= take

                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])

    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [ ]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import callbacks, layers
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import product
import tensorflow_probability as tfp

class MotionCodeModel(tf.keras.Model):
    def __init__(
        self,
        num_processes,
        num_timestamps=10,
        latent_dim=16,
        noise_variance=0.1,
        lambda_reg=1e-3,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.L = int(num_processes)
        self.m = int(num_timestamps)
        self.d = int(latent_dim)
        self.lambda_reg = float(lambda_reg)

        self.sigma_sq = self.add_weight(
            name="sigma_sq",
            shape=(),
            initializer=tf.keras.initializers.Constant(float(noise_variance)),
            trainable=True,
            dtype=tf.float32
        )

        initial_theta = np.linspace(0.1, 0.9, self.m, dtype=np.float32)
        initial_theta = np.tile(initial_theta[:, np.newaxis], (1, self.d))
        self.theta = self.add_weight(
            name="theta_mapping",
            shape=initial_theta.shape,
            initializer=tf.keras.initializers.Constant(initial_theta),
            trainable=True,
            dtype=tf.float32
        )

        self.motion_codes = self.add_weight(
            name="motion_codes_z",
            shape=(self.L, self.d),
            initializer="ones",
            trainable=True,
            dtype=tf.float32
        )

        self.kernel_eta = self.add_weight(
            name="kernel_eta",
            shape=(self.L, 2),
            initializer="zeros",
            trainable=True,
            dtype=tf.float32
        )

    def compute_kernel(self, t1, t2, eta_k):
        amplitude = tf.exp(eta_k[0])
        lengthscale = tf.exp(eta_k[1]) + 1e-5

        t1 = tf.reshape(t1, (-1, 1))
        t2 = tf.reshape(tf.transpose(t2), (1, -1))

        sq_dist = tf.square(t1 - t2)
        kernel_mat = amplitude * tf.exp(-0.5 * sq_dist / tf.square(lengthscale))
        return kernel_mat

    def get_most_informative_timestamps(self):
        return tf.sigmoid(tf.matmul(self.motion_codes, self.theta, transpose_b=True))

    def compute_l_max_approx(self, C_k, T_i_list, eta_k, s_m_k):
        B = len(T_i_list)
        jitter = 1e-4

        s_m_k = tf.reshape(s_m_k, (-1,))
        K_SmSm = self.compute_kernel(s_m_k, s_m_k, eta_k) + jitter * tf.eye(self.m, dtype=tf.float32)
        L_sm = tf.linalg.cholesky(K_SmSm)

        trace_sum = 0.0
        q_blocks = []
        y_list = []

        for i, T_i in enumerate(T_i_list):
            T_i = tf.reshape(T_i, (-1,))
            y_i = tf.reshape(C_k[i], (-1,))
            y_list.append(y_i)

            K_TiTi = self.compute_kernel(T_i, T_i, eta_k)
            K_SmTi = self.compute_kernel(s_m_k, T_i, eta_k)
            K_TiSm = tf.transpose(K_SmTi)

            sol = tf.linalg.cholesky_solve(L_sm, K_SmTi)
            Q_TiTi = tf.matmul(K_TiSm, sol)
            q_blocks.append(Q_TiTi)

            trace_sum += tf.reduce_sum(tf.linalg.diag_part(K_TiTi - Q_TiTi))

        Y = tf.concat(y_list, axis=0)
        QC_G = tf.linalg.diag(tf.concat([tf.linalg.diag_part(qb) for qb in q_blocks], axis=0))

        total_points = tf.shape(Y)[0]
        cov_matrix = (float(B) * self.sigma_sq * tf.eye(total_points, dtype=tf.float32)) + QC_G + jitter * tf.eye(total_points, dtype=tf.float32)

        mvn = tfp.distributions.MultivariateNormalTriL(
            loc=tf.zeros((total_points,), dtype=tf.float32),
            scale_tril=tf.linalg.cholesky(cov_matrix)
        )
        log_p_y = mvn.log_prob(Y)
        term2 = trace_sum / (2.0 * self.sigma_sq * float(B))
        return log_p_y - term2

    def compute_total_loss(self, data_collections, timestamps_collections):
        S_m = self.get_most_informative_timestamps()
        total_l_max = 0.0

        for k in range(self.L):
            eta_k = self.kernel_eta[k]
            s_m_k = S_m[k]
            C_k = data_collections[k]
            T_i_list = timestamps_collections[k]

            l_max_k = self.compute_l_max_approx(C_k, T_i_list, eta_k, s_m_k)
            total_l_max += l_max_k

        reg_term = self.lambda_reg * tf.reduce_sum(tf.square(self.motion_codes))
        return -total_l_max + reg_term

    def forecast(self, k, T_target, C_k, T_i_list):
        eta_k = self.kernel_eta[k]
        s_m_k = self.get_most_informative_timestamps()[k]

        T_target = tf.reshape(T_target, (-1, 1))
        s_m_k = tf.reshape(s_m_k, (-1, 1))

        K_SmSm = self.compute_kernel(s_m_k, s_m_k, eta_k) + 1e-4 * tf.eye(self.m, dtype=tf.float32)
        K_SmT_target = self.compute_kernel(s_m_k, T_target, eta_k)

        # Ensure Y_k is reshaped properly as a column vector (N x 1)
        Y_k = tf.concat([tf.reshape(y_i, (-1, 1)) for y_i in C_k], axis=0)
        T_k = tf.concat([tf.reshape(T_i, (-1, 1)) for T_i in T_i_list], axis=0)

        K_TK_Sm = self.compute_kernel(T_k, s_m_k, eta_k)
        K_T_T_train = self.compute_kernel(T_k, T_k, eta_k) + (self.sigma_sq + 1e-4) * tf.eye(tf.shape(T_k)[0], dtype=tf.float32)

        cov_inv_k = tf.linalg.cholesky_solve(tf.linalg.cholesky(K_T_T_train), K_TK_Sm)
        inv_expression = tf.linalg.inv(K_SmSm + tf.matmul(tf.transpose(K_TK_Sm), cov_inv_k))

        intermediate = tf.matmul(inv_expression, tf.transpose(cov_inv_k))
        mean_signal = tf.matmul(tf.matmul(tf.transpose(K_SmT_target), intermediate), Y_k)
        return tf.reshape(mean_signal, (-1,))

    def classify(self, y_query, T_query, data_collections, timestamps_collections):
        best_k = 0
        min_dist = float('inf')

        for k in range(self.L):
            p_k = self.forecast(k, T_query, data_collections[k], timestamps_collections[k])
            y_q = tf.reshape(y_query, (-1,))
            p_k_res = tf.reshape(p_k, (-1,))

            min_len = min(tf.shape(y_q)[0], tf.shape(p_k_res)[0])
            dist = tf.norm(y_q[:min_len] - p_k_res[:min_len], ord=2)
            if dist < min_dist:
                min_dist = dist
                best_k = k

        return best_k

In [ ]:
# ==========================================
# Motion Code Data Preparation Helpers
# ==========================================

def prepare_motion_code_collections(
    hc_sub_list,
    pd_sub_list,
    max_realizations=5,
    max_timepoints=32
):
    """
    Converts subject arrays into Motion Code collections.

    Each class/process is represented by a small collection
    of time-series realizations to keep GP matrix operations
    computationally manageable.
    """

    collections = []
    timestamps_list = []

    for sub_group in [hc_sub_list, pd_sub_list]:

        C_k = []
        T_i_k = []

        for sub in sub_group:

            arr = np.asarray(sub, dtype=np.float32)

            if arr.ndim > 2:
                arr = arr.reshape(-1, arr.shape[-1])

            elif arr.ndim == 1:
                arr = arr[np.newaxis, :]

            for time_series in arr:

                time_series = np.asarray(
                    time_series,
                    dtype=np.float32
                )

                # ------------------------------------------
                # Downsample long time series
                # ------------------------------------------
                if len(time_series) > max_timepoints:

                    indices = np.linspace(
                        0,
                        len(time_series) - 1,
                        max_timepoints,
                        dtype=int
                    )

                    time_series = time_series[indices]

                # ------------------------------------------
                # Store realization
                # ------------------------------------------
                C_k.append(time_series)

                # Normalized time axis [0, 1]
                T_i_k.append(
                    np.linspace(
                        0.0,
                        1.0,
                        len(time_series),
                        dtype=np.float32
                    )
                )

        # ----------------------------------------------
        # Limit number of realizations for OOM protection
        # ----------------------------------------------
        if len(C_k) > max_realizations:

            indices = np.random.choice(
                len(C_k),
                max_realizations,
                replace=False
            )

            C_k = [
                C_k[i]
                for i in indices
            ]

            T_i_k = [
                T_i_k[i]
                for i in indices
            ]

        collections.append(C_k)
        timestamps_list.append(T_i_k)

    return collections, timestamps_list


def format_query_series(
    sub,
    max_timepoints=32
):
    """
    Safely converts a subject/query into a single
    Motion Code time series and normalized time axis.
    """

    arr = np.asarray(
        sub,
        dtype=np.float32
    )

    query_series = arr.flatten()

    # ------------------------------------------
    # Downsample query if necessary
    # ------------------------------------------
    if len(query_series) > max_timepoints:

        indices = np.linspace(
            0,
            len(query_series) - 1,
            max_timepoints,
            dtype=int
        )

        query_series = query_series[indices]

    # Normalized time axis [0, 1]
    query_time = np.linspace(
        0.0,
        1.0,
        len(query_series),
        dtype=np.float32
    )

    return query_series, query_time

In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from itertools import product

def run_subject_level_mc_cv_optimized(X_healthy, X_pd, SEED=42, epochs=50):
    """
    Nested cross-validation pipeline for Motion Code,
    fully structured and synchronized without validation/early stopping.
    """
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # ==========================================
    # Motion Code Hyperparameter Grid Search
    # ==========================================
    param_grid = {
        'lr': [1e-2, 1e-3],
        'batch_size': [32]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    # ==========================================
    # OUTER CROSS-VALIDATION LOOP
    # ==========================================
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 (MOTION CODE) ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        # ==========================================
        # INNER CV HYPERPARAMETER SELECTION
        # ==========================================
        for params in all_combinations:
            inner_fold_accuracies = []

            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # Prepare Inner Collections
                train_data_cols, train_time_cols = prepare_motion_code_collections(hc_train_sub, pd_train_sub)

                # Instantiate Inner Motion Code Model
                inner_model = MotionCodeModel(
                    num_processes=2,
                    num_timestamps=10,
                    latent_dim=16
                )
                optimizer = tf.keras.optimizers.Adam(learning_rate=params['lr'])

                # Fixed-Epoch Inner Training (No Validation/Early Stopping)
                for _ in range(epochs):
                    with tf.GradientTape() as tape:
                        loss = inner_model.compute_total_loss(train_data_cols, train_time_cols)
                    grads = tape.gradient(loss, inner_model.trainable_variables)
                    optimizer.apply_gradients(zip(grads, inner_model.trainable_variables))

                # Inner Validation Phase (Evaluation Only)
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_predictions = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    if np.asarray(sub).size == 0:
                        continue

                    query_series, query_time = format_query_series(sub)
                    pred_class = inner_model.classify(
                        query_series, query_time, train_data_cols, train_time_cols
                    )
                    val_predictions.append(int(pred_class))
                    valid_val_labels.append(int(true_lbl))

                inner_acc = accuracy_score(valid_val_labels, val_predictions) if valid_val_labels else 0.0
                inner_fold_accuracies.append(inner_acc)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params

        print(f">> Best Motion Code Grid Parameters Selected: {best_params} | Inner Acc: {best_score:.4f}")

        # ==========================================
        # OUTER TRAINING & FINAL FIT (FIXED EPOCHS)
        # ==========================================
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        final_train_data_cols, final_train_time_cols = prepare_motion_code_collections(hc_train_final, pd_train_final)

        final_model = MotionCodeModel(
            num_processes=2,
            num_timestamps=10,
            latent_dim=16
        )
        final_optimizer = tf.keras.optimizers.Adam(learning_rate=best_params['lr'])

        # Track Training Time (Fixed Epochs)
        start_time = time.time()
        for epoch in range(epochs):
            with tf.GradientTape() as tape:
                loss = final_model.compute_total_loss(final_train_data_cols, final_train_time_cols)
            grads = tape.gradient(loss, final_model.trainable_variables)
            final_optimizer.apply_gradients(zip(grads, final_model.trainable_variables))

        training_time = time.time() - start_time

        # Track Model Parameters Count
        model_params_count = sum(np.prod(v.shape) for v in final_model.trainable_variables)

        # ==========================================
        # OUTER TEST SET EVALUATION
        # ==========================================
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        for sub, true_label in zip(test_subjects, test_labels):
            if np.asarray(sub).size == 0:
                continue

            query_series, query_time = format_query_series(sub)
            pred = final_model.classify(
                query_series, query_time, final_train_data_cols, final_train_time_cols
            )
            pred = int(pred)

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        # ==========================================
        # Summary Metrics Record
        # ==========================================
        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': "N/A (Direct Class)",
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}",
            'Training Time (s)': f"{training_time:.2f}",
            'Model Parameters': f"{model_params_count:,}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}% | Time: {training_time:.2f}s | Params: {model_params_count:,}")

    # ==========================================
    # FINAL SUMMARY REPORT
    # ==========================================
    summary_df = pd.DataFrame(fold_summary_records)
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy (Motion Code): {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [ ]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


In [ ]:
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)


========== OUTER FOLD 1 / 5 (MOTION CODE) ==========


I0000 00:00:1788951174.664071      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788951174.667149      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1788951175.600905      58 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x6b0a1dc0


>> Best Motion Code Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32} | Inner Acc: 0.5434
Outer Fold 1 Stats -> Healthy: 3/6 | PD: 12/24 | Acc: 50.00% | Time: 15.74s | Params: 197.0

========== OUTER FOLD 2 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32} | Inner Acc: 0.5043
Outer Fold 2 Stats -> Healthy: 1/6 | PD: 14/23 | Acc: 51.72% | Time: 15.98s | Params: 197.0

========== OUTER FOLD 3 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32} | Inner Acc: 0.5304
Outer Fold 3 Stats -> Healthy: 5/6 | PD: 13/23 | Acc: 62.07% | Time: 16.02s | Params: 197.0

========== OUTER FOLD 4 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32} | Inner Acc: 0.5000
Outer Fold 4 Stats -> Healthy: 1/5 | PD: 13/23 | Acc: 50.00% | Time: 15.98s | Params: 197.0

========== OUTER FOLD 5 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid

In [ ]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/3059442721.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/3059442721.py:41: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in 


========== OUTER FOLD 1 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.01, 'batch_size': 32} | Inner Acc: 0.5524
Outer Fold 1 Stats -> Healthy: 2/5 | PD: 9/22 | Acc: 40.74% | Time: 15.94s | Params: 197.0

========== OUTER FOLD 2 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.01, 'batch_size': 32} | Inner Acc: 0.5000
Outer Fold 2 Stats -> Healthy: 4/5 | PD: 12/22 | Acc: 59.26% | Time: 15.95s | Params: 197.0

========== OUTER FOLD 3 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.01, 'batch_size': 32} | Inner Acc: 0.5325
Outer Fold 3 Stats -> Healthy: 4/5 | PD: 10/22 | Acc: 51.85% | Time: 15.91s | Params: 197.0

========== OUTER FOLD 4 / 5 (MOTION CODE) ==========
>> Best Motion Code Grid Parameters Selected: {'lr': 0.01, 'batch_size': 32} | Inner Acc: 0.5056
Outer Fold 4 Stats -> Healthy: 4/4 | PD: 13/22 | Acc: 65.38% | Time: 15.91s | Params: 197.0

========== OUTER FOLD 5 / 5 